# 🧭 Customer Feedback Clustering & Product Roadmap Agent

**Problem:** Product managers are flooded with hundreds of feature requests from many channels, which makes patterns hard to see.

**Agent:** Claude (`claude-opus-5-5`) reads every piece of feedback, **categorizes** it, **clusters** it by underlying need, and the code **counts and scores** each cluster (volume, distinct accounts, ARR, channel breadth, urgency) to surface the most-demanded updates as a Now / Next / Later roadmap.

**Architecture**
```
feedback CSV ─▶ Claude agent loop (tool runner) ──tools──▶ FeedbackStore
                 read_feedback · create_cluster · assign_feedback    (counts & scores
                 merge_clusters · get_unassigned · get_priorities     computed in code,
                 submit_roadmap                                        never by the LLM)
                                   │
                                   ▼
                     Streamlit dashboard  ·  Markdown / HTML / JSON report
```

### Before you start
1. Open the **🔑 Secrets** panel on the left of Colab and add a secret named `ANTHROPIC_API_KEY`, then turn on notebook access.
   *(No key? Everything still runs with the offline keyword baseline.)*
2. **Runtime → Run all**, then open the link printed by the last cell.

## 1 · Install dependencies

In [ ]:
!pip install -q 'anthropic>=1.13.0' 'streamlit>=1.40' pandas altair

## 2 · Write the project files
Each cell below writes one module. Edit them here and re-run to change the agent.

In [ ]:
!mkdir -p roadmap_agent data output

In [ ]:
%%writefile roadmap_agent/__init__.py

**Feedback store** – loading, clusters, and the deterministic priority score.

In [ ]:
%%writefile roadmap_agent/store.py
"""Feedback store: loads raw feedback, holds clusters, and computes demand scores.

The LLM decides *what* belongs together (semantic judgment); this module does
all the *counting and scoring* deterministically, so numbers in the roadmap are
never hallucinated.
"""

from __future__ import annotations

import csv
import json
from dataclasses import dataclass, field
from datetime import date
from pathlib import Path

CATEGORIES = [
    "feature_request",
    "integration",
    "security_compliance",
    "performance",
    "ux_improvement",
    "bug",
    "not_a_request",  # praise, billing questions, etc. - excluded from the roadmap
]

# Categories that represent product work worth putting on a roadmap.
ROADMAP_CATEGORIES = {"feature_request", "integration", "security_compliance", "performance", "ux_improvement"}

# Weights for the priority score (sum to 1.0). Tune these to your strategy.
WEIGHTS = {
    "requests": 0.30,        # raw volume of mentions
    "accounts": 0.25,        # distinct customers asking
    "arr": 0.25,             # revenue of customers asking
    "channels": 0.10,        # breadth: asked for across many channels
    "urgency": 0.10,         # share of mentions flagged as blocking / churn risk
}


@dataclass
class Feedback:
    id: str
    date: str
    channel: str
    customer: str
    plan: str
    arr: float
    text: str


@dataclass
class Cluster:
    id: str
    name: str
    description: str
    category: str
    feedback_ids: list[str] = field(default_factory=list)
    urgent_ids: set[str] = field(default_factory=set)


class FeedbackStore:
    def __init__(self, feedback: list[Feedback]):
        self.feedback = {f.id: f for f in feedback}
        self.clusters: dict[str, Cluster] = {}
        self.assignment: dict[str, str] = {}  # feedback_id -> cluster_id
        self._next = 1

    # ---------- loading ----------
    @classmethod
    def from_file(cls, path: str | Path) -> "FeedbackStore":
        path = Path(path)
        if path.suffix == ".json":
            rows = json.loads(path.read_text())
        else:
            with path.open(newline="", encoding="utf-8") as fh:
                rows = list(csv.DictReader(fh))
        items = []
        for i, r in enumerate(rows, 1):
            items.append(
                Feedback(
                    id=str(r.get("id") or f"F{i:04d}"),
                    date=r.get("date", ""),
                    channel=r.get("channel", "unknown"),
                    customer=r.get("customer", "unknown"),
                    plan=r.get("plan", ""),
                    arr=float(r.get("arr") or 0),
                    text=r["text"].strip(),
                )
            )
        return cls(items)

    # ---------- cluster operations (exposed to the agent as tools) ----------
    def create_cluster(self, name: str, description: str, category: str) -> Cluster:
        if category not in CATEGORIES:
            raise ValueError(f"category must be one of {CATEGORIES}")
        for c in self.clusters.values():
            if c.name.lower() == name.lower():
                raise ValueError(f"cluster '{name}' already exists as {c.id}")
        cid = f"C{self._next:02d}"
        self._next += 1
        self.clusters[cid] = Cluster(cid, name, description, category)
        return self.clusters[cid]

    def assign(self, cluster_id: str, feedback_ids: list[str], urgent_ids: list[str] | None = None) -> dict:
        if cluster_id not in self.clusters:
            raise ValueError(f"unknown cluster {cluster_id}")
        unknown = [f for f in feedback_ids if f not in self.feedback]
        moved = []
        for fid in feedback_ids:
            if fid not in self.feedback:
                continue
            prev = self.assignment.get(fid)
            if prev and prev != cluster_id:
                self.clusters[prev].feedback_ids.remove(fid)
                self.clusters[prev].urgent_ids.discard(fid)
                moved.append(fid)
            if fid not in self.clusters[cluster_id].feedback_ids:
                self.clusters[cluster_id].feedback_ids.append(fid)
            self.assignment[fid] = cluster_id
        for fid in urgent_ids or []:
            if self.assignment.get(fid) == cluster_id:
                self.clusters[cluster_id].urgent_ids.add(fid)
        return {"cluster": cluster_id, "size": len(self.clusters[cluster_id].feedback_ids),
                "moved_from_other_clusters": moved, "unknown_ids": unknown}

    def merge(self, source_id: str, target_id: str) -> dict:
        src, tgt = self.clusters.get(source_id), self.clusters.get(target_id)
        if not src or not tgt or source_id == target_id:
            raise ValueError("need two distinct, existing cluster ids")
        urgent = list(src.urgent_ids)
        self.assign(target_id, list(src.feedback_ids), urgent)
        del self.clusters[source_id]
        return {"merged_into": target_id, "size": len(tgt.feedback_ids)}

    def unassigned(self) -> list[str]:
        return [fid for fid in self.feedback if fid not in self.assignment]

    # ---------- analytics ----------
    def cluster_stats(self, c: Cluster) -> dict:
        items = [self.feedback[f] for f in c.feedback_ids]
        accounts = {i.customer for i in items}
        # ARR counted once per account, not once per mention.
        arr = sum({i.customer: i.arr for i in items}.values())
        dates = sorted(i.date for i in items if i.date)
        return {
            "id": c.id,
            "name": c.name,
            "category": c.category,
            "description": c.description,
            "requests": len(items),
            "accounts": len(accounts),
            "arr": arr,
            "channels": sorted({i.channel for i in items}),
            "urgent": len(c.urgent_ids),
            "first_seen": dates[0] if dates else "",
            "last_seen": dates[-1] if dates else "",
            "quotes": [f"[{i.id} · {i.channel} · {i.customer}] {i.text}" for i in items[:3]],
        }

    def prioritize(self) -> list[dict]:
        """Rank roadmap-worthy clusters with a transparent weighted score (0-100)."""
        stats = [self.cluster_stats(c) for c in self.clusters.values()
                 if c.category in ROADMAP_CATEGORIES and c.feedback_ids]
        if not stats:
            return []
        n_channels = len({f.channel for f in self.feedback.values()}) or 1
        mx = {k: max(s[k] for s in stats) or 1 for k in ("requests", "accounts", "arr")}
        for s in stats:
            parts = {
                "requests": s["requests"] / mx["requests"],
                "accounts": s["accounts"] / mx["accounts"],
                "arr": s["arr"] / mx["arr"],
                "channels": len(s["channels"]) / n_channels,
                "urgency": s["urgent"] / s["requests"],
            }
            s["score"] = round(100 * sum(WEIGHTS[k] * v for k, v in parts.items()), 1)
        stats.sort(key=lambda s: s["score"], reverse=True)
        for rank, s in enumerate(stats, 1):
            s["rank"] = rank
            s["tier"] = "Now" if s["score"] >= 60 else "Next" if s["score"] >= 35 else "Later"
        return stats

    def summary(self) -> dict:
        by_channel: dict[str, int] = {}
        for f in self.feedback.values():
            by_channel[f.channel] = by_channel.get(f.channel, 0) + 1
        return {
            "total_feedback": len(self.feedback),
            "assigned": len(self.assignment),
            "unassigned": len(self.unassigned()),
            "clusters": len(self.clusters),
            "by_channel": by_channel,
            "generated": date.today().isoformat(),
        }

**The Claude agent** – system prompt + tools; the SDK tool runner drives the loop.

In [ ]:
%%writefile roadmap_agent/agent.py
"""The Claude-powered clustering & roadmap agent.

Claude drives the loop (via the SDK tool runner): it reads feedback in batches,
creates/merges clusters, assigns every item, asks for the deterministic
priority ranking, and finally submits a roadmap with its rationale.
"""

from __future__ import annotations

import json

import anthropic
from anthropic import beta_tool

from .store import CATEGORIES, FeedbackStore

MODEL = "claude-opus-5-5"

SYSTEM_PROMPT = f"""You are a senior product-operations analyst. Product managers are flooded with
feature requests from many channels (app store reviews, support tickets, sales calls, NPS surveys,
social media, community forums). Your job is to turn that raw feedback into a clear, evidence-backed
product roadmap.

Work through it with your tools:
1. Call get_overview, then read_feedback in batches until you have seen every item.
2. Group items by the underlying user NEED, not surface wording ("night mode", "dark theme" and
   "white background is blinding" are one cluster; "SSO" and "SCIM provisioning" are related but
   distinct asks). Prefer specific, actionable clusters over vague buckets like "general requests".
   Categories: {", ".join(CATEGORIES)}. Use `bug` for defects and crashes, and `not_a_request` for
   praise, billing questions and similar items, so they stay out of the roadmap.
3. When assigning, pass urgent_ids for items that signal churn, a blocked deal or renewal, a
   compliance requirement, or a customer abandoning the product.
4. Merge clusters that turn out to be duplicates. Keep going until get_unassigned returns nothing.
5. Call get_priorities. The scores are computed in code from request volume, distinct accounts,
   revenue, channel breadth and urgency. Do not invent numbers; cite the ones the tool returns.
6. Call submit_roadmap once with an executive summary and one rationale per roadmap cluster.
   Note where the score alone is misleading (e.g. one huge account, or a cheap quick win)."""


def build_tools(store: FeedbackStore, result: dict):
    @beta_tool
    def get_overview() -> str:
        """Get totals: number of feedback items, how many are clustered, and counts per channel."""
        return json.dumps(store.summary())

    @beta_tool
    def read_feedback(offset: int = 0, limit: int = 40) -> str:
        """Read raw feedback items in a batch.

        Args:
            offset: Index of the first item to return.
            limit: Maximum number of items to return (max 60).
        """
        items = list(store.feedback.values())[offset: offset + min(limit, 60)]
        rows = [{"id": f.id, "channel": f.channel, "customer": f.customer, "plan": f.plan,
                 "text": f.text} for f in items]
        return json.dumps({"offset": offset, "returned": len(rows),
                           "total": len(store.feedback), "items": rows})

    @beta_tool
    def list_clusters() -> str:
        """List current clusters with their size and a few member ids."""
        return json.dumps([{"id": c.id, "name": c.name, "category": c.category,
                            "description": c.description, "size": len(c.feedback_ids),
                            "sample_ids": c.feedback_ids[:5]} for c in store.clusters.values()])

    @beta_tool
    def create_cluster(name: str, description: str, category: str) -> str:
        """Create a new cluster representing one underlying user need.

        Args:
            name: Short, specific name, e.g. "SAML SSO (Okta / Azure AD)".
            description: One sentence describing the need.
            category: One of feature_request, integration, security_compliance, performance,
                ux_improvement, bug, not_a_request.
        """
        try:
            c = store.create_cluster(name, description, category)
        except ValueError as e:
            return f"ERROR: {e}"
        return json.dumps({"id": c.id, "name": c.name})

    @beta_tool
    def assign_feedback(cluster_id: str, feedback_ids: list[str], urgent_ids: list[str] | None = None) -> str:
        """Assign feedback items to a cluster. Re-assigning an item moves it.

        Args:
            cluster_id: Target cluster id, e.g. "C03".
            feedback_ids: Feedback ids to put in this cluster.
            urgent_ids: Subset of feedback_ids signalling churn risk, a blocked deal, or a compliance need.
        """
        try:
            return json.dumps(store.assign(cluster_id, feedback_ids, urgent_ids))
        except ValueError as e:
            return f"ERROR: {e}"

    @beta_tool
    def merge_clusters(source_id: str, target_id: str) -> str:
        """Merge a duplicate cluster into another one; the source cluster is deleted.

        Args:
            source_id: Cluster to fold in and remove.
            target_id: Cluster that keeps all items.
        """
        try:
            return json.dumps(store.merge(source_id, target_id))
        except ValueError as e:
            return f"ERROR: {e}"

    @beta_tool
    def get_unassigned() -> str:
        """Return the full text of any feedback items not yet assigned to a cluster."""
        ids = store.unassigned()
        return json.dumps({"count": len(ids), "items": [
            {"id": i, "text": store.feedback[i].text} for i in ids]})

    @beta_tool
    def get_priorities() -> str:
        """Compute the ranked roadmap: score (0-100), tier (Now/Next/Later), requests, accounts, ARR, channels."""
        ranked = store.prioritize()
        return json.dumps([{k: s[k] for k in ("rank", "id", "name", "tier", "score", "requests",
                                                "accounts", "arr", "channels", "urgent")} for s in ranked])

    @beta_tool
    def submit_roadmap(executive_summary: str, rationales: dict[str, str]) -> str:
        """Submit the final roadmap. Call exactly once, after get_priorities.

        Args:
            executive_summary: 3-5 sentences for a product leader: the top themes, what to build now, and key risks.
            rationales: Map of cluster id -> 1-2 sentence rationale grounded in the numbers and quotes.
        """
        if store.unassigned():
            return f"ERROR: {len(store.unassigned())} items are still unassigned; call get_unassigned."
        result["executive_summary"] = executive_summary
        result["rationales"] = rationales
        return "Roadmap saved."

    return [get_overview, read_feedback, list_clusters, create_cluster, assign_feedback,
            merge_clusters, get_unassigned, get_priorities, submit_roadmap]


def run_agent(store: FeedbackStore, on_event=print, api_key: str | None = None) -> dict:
    """Run the agent to completion. `on_event(str)` receives a log line per step (for CLI or UI)."""
    client = anthropic.Anthropic(api_key=api_key) if api_key else anthropic.Anthropic()
    result: dict = {}
    runner = client.beta.messages.tool_runner(
        model=MODEL,
        max_tokens=16000,
        system=SYSTEM_PROMPT,
        thinking={"type": "adaptive"},
        output_config={"effort": "high"},
        # Server-side fallback if a request is ever declined by safety classifiers.
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        max_iterations=60,
        tools=build_tools(store, result),
        messages=[{"role": "user", "content":
                   "Cluster all of the customer feedback and produce a prioritized product roadmap."}],
    )
    for message in runner:
        if message.stop_reason == "refusal":
            raise RuntimeError(f"Request declined: {message.stop_details}")
        for block in message.content:
            if block.type == "tool_use":
                args = json.dumps(block.input)
                on_event(f"→ {block.name}({args[:100]}{'…' if len(args) > 100 else ''})")
            elif block.type == "text" and block.text.strip():
                on_event(block.text.strip()[:300])
    if "executive_summary" not in result:
        raise RuntimeError("Agent finished without submitting a roadmap.")
    return result

**Offline baseline** – keyword rules, no API key needed.

In [ ]:
%%writefile roadmap_agent/offline.py
"""Keyword-rule baseline: runs with no API key.

Useful for demos, tests, and as a comparison point that shows what the LLM
agent adds (it handles paraphrases and new themes these rules cannot).
"""

from __future__ import annotations

import re

from .store import FeedbackStore

# (name, category, description, keyword regex) - first match wins, so order matters.
RULES = [
    ("Crashes & freezes", "bug", "App crashes or freezes.", r"crash|freez"),
    ("Billing & account questions", "not_a_request", "Billing/account support, not product asks.", r"charged|refund|billing"),
    ("Praise", "not_a_request", "Positive feedback with no request.", r"^(great|love the|your onboarding)|keep it up"),
    ("SCIM user provisioning", "security_compliance", "Automatic user provisioning from the IdP.", r"scim|provisioning"),
    ("Single sign-on (SAML / Okta / Azure AD / Google)", "security_compliance", "Enterprise SSO login.", r"\bsso\b|single sign-on|saml|okta"),
    ("Two-factor authentication", "security_compliance", "MFA / 2FA, enforceable by admins.", r"2fa|mfa|two-factor"),
    ("Audit logs", "security_compliance", "Exportable log of admin actions.", r"audit log"),
    ("Dark mode", "ux_improvement", "Dark / night theme.", r"dark|night mode|white background"),
    ("Slack & Microsoft Teams integration", "integration", "Notifications and actions in chat tools.", r"slack|teams"),
    ("Offline mode (mobile)", "feature_request", "Use the app without connectivity.", r"offline|on flights|without signal|subway"),
    ("Public API, webhooks & rate limits", "integration", "Programmatic access and automation.", r"\bapi\b|webhook|rate limit"),
    ("Custom dashboards & report builder", "feature_request", "User-defined dashboards and reports.", r"dashboard|report|custom charts"),
    ("Data export (CSV / Excel / PDF)", "feature_request", "Export data in common formats.", r"export|csv|excel|pdf"),
    ("Recurring tasks & templates", "feature_request", "Repeat tasks and reusable templates.", r"recurring|repeat|template|reminder"),
    ("Localization", "feature_request", "UI in more languages.", r"spanish|portuguese|german|french|locali|translat"),
    ("Speed & performance", "performance", "Slow load times and sluggish UI.", r"slow|sluggish|forever|faster|performance|seconds"),
]

URGENT = r"block|deal|dealbreaker|renewal|risk|lost the|must-have|requires|required|contingent|auditor|compliance|deleting|ciso"


def cluster_offline(store: FeedbackStore) -> dict:
    ids: dict[str, str] = {}
    other = None
    for f in store.feedback.values():
        text = f.text.lower()
        for name, cat, desc, pattern in RULES:
            if re.search(pattern, text):
                if name not in ids:
                    ids[name] = store.create_cluster(name, desc, cat).id
                urgent = [f.id] if re.search(URGENT, text) else []
                store.assign(ids[name], [f.id], urgent)
                break
        else:
            other = other or store.create_cluster("Other / uncategorized", "No rule matched.", "feature_request").id
            store.assign(other, [f.id])
    ranked = store.prioritize()
    top = ", ".join(s["name"] for s in ranked[:3])
    return {
        "executive_summary": f"(Offline keyword baseline - run with an API key for the Claude agent's analysis.) "
                             f"Highest-demand themes: {top}.",
        "rationales": {},
    }

**Reports** – Markdown, HTML dashboard and JSON.

In [ ]:
%%writefile roadmap_agent/report.py
"""Render the roadmap as Markdown, JSON and a self-contained HTML dashboard."""

from __future__ import annotations

import json
from html import escape
from pathlib import Path

from .store import ROADMAP_CATEGORIES, FeedbackStore


def build(store: FeedbackStore, result: dict) -> dict:
    ranked = store.prioritize()
    other = [store.cluster_stats(c) for c in store.clusters.values()
             if c.category not in ROADMAP_CATEGORIES and c.feedback_ids]
    for s in ranked:
        s["rationale"] = result.get("rationales", {}).get(s["id"], "")
    return {"summary": store.summary(), "executive_summary": result.get("executive_summary", ""),
            "roadmap": ranked, "excluded": other}


def write_all(report: dict, out_dir: str | Path) -> list[Path]:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    paths = [out / "roadmap.json", out / "roadmap.md", out / "roadmap.html"]
    paths[0].write_text(json.dumps(report, indent=2))
    paths[1].write_text(to_markdown(report))
    paths[2].write_text(to_html(report))
    return paths


def money(x: float) -> str:
    return f"${x / 1000:,.0f}k" if x >= 1000 else f"${x:,.0f}"


def to_markdown(r: dict) -> str:
    s = r["summary"]
    lines = [
        "# Product Roadmap from Customer Feedback", "",
        f"_{s['total_feedback']} feedback items · {len(s['by_channel'])} channels · "
        f"{s['clusters']} clusters · generated {s['generated']}_", "",
        "## Executive summary", "", r["executive_summary"], "",
        "## Prioritized roadmap", "",
        "| # | Tier | Theme | Score | Requests | Accounts | ARR asking | Channels | Urgent |",
        "|---|------|-------|------:|---------:|---------:|-----------:|---------:|-------:|",
    ]
    for x in r["roadmap"]:
        lines.append(f"| {x['rank']} | {x['tier']} | {x['name']} | {x['score']} | {x['requests']} | "
                     f"{x['accounts']} | {money(x['arr'])} | {len(x['channels'])} | {x['urgent']} |")
    lines += ["", "## Theme details", ""]
    for x in r["roadmap"]:
        lines += [f"### {x['rank']}. {x['name']} — {x['tier']} (score {x['score']})", "",
                  f"*{x['category']}* · {x['description']}", ""]
        if x.get("rationale"):
            lines += [f"**Why:** {x['rationale']}", ""]
        lines += [f"> {q}" for q in x["quotes"]] + [""]
    if r["excluded"]:
        lines += ["## Not on the roadmap (bugs & non-requests)", ""]
        lines += [f"- **{x['name']}** ({x['category']}): {x['requests']} items" for x in r["excluded"]]
    return "\n".join(lines) + "\n"


def to_html(r: dict) -> str:
    s = r["summary"]
    top = max((x["score"] for x in r["roadmap"]), default=1) or 1
    rows = []
    for x in r["roadmap"]:
        quotes = "".join(f"<li>{escape(q)}</li>" for q in x["quotes"])
        why = f"<p class='why'>{escape(x['rationale'])}</p>" if x.get("rationale") else ""
        rows.append(f"""
<details class="item">
  <summary>
    <span class="rank">{x['rank']}</span>
    <span class="tier t-{x['tier'].lower()}">{x['tier']}</span>
    <span class="name">{escape(x['name'])}</span>
    <span class="bar"><span style="width:{100 * x['score'] / top:.0f}%"></span></span>
    <span class="score">{x['score']}</span>
  </summary>
  <div class="body">
    <p class="meta">{x['requests']} requests · {x['accounts']} accounts · {money(x['arr'])} ARR asking ·
      {x['urgent']} urgent · {escape(', '.join(x['channels']))}</p>
    {why}
    <ul class="quotes">{quotes}</ul>
  </div>
</details>""")
    excluded = "".join(f"<li>{escape(x['name'])} <span class='muted'>({x['category']}, {x['requests']})</span></li>"
                       for x in r["excluded"])
    channels = " · ".join(f"{escape(k)} {v}" for k, v in sorted(s["by_channel"].items(), key=lambda kv: -kv[1]))
    return f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Feedback Roadmap</title>
<style>
:root {{ --bg:#fafaf9; --card:#fff; --ink:#1c1917; --muted:#78716c; --line:#e7e5e4; --accent:#2563eb;
  --now:#dc2626; --next:#d97706; --later:#64748b; }}
@media (prefers-color-scheme: dark) {{ :root {{ --bg:#0c0a09; --card:#1c1917; --ink:#f5f5f4; --muted:#a8a29e;
  --line:#292524; --accent:#60a5fa; --now:#f87171; --next:#fbbf24; --later:#94a3b8; }} }}
* {{ box-sizing:border-box }}
body {{ margin:0; background:var(--bg); color:var(--ink); font:15px/1.5 system-ui,-apple-system,sans-serif }}
main {{ max-width:900px; margin:0 auto; padding:32px 16px }}
h1 {{ margin:0 0 4px; font-size:26px }} h2 {{ font-size:17px; margin:32px 0 12px }}
.muted, .meta {{ color:var(--muted) }}
.kpis {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(140px,1fr)); gap:12px; margin:20px 0 }}
.kpi, .summary, .item {{ background:var(--card); border:1px solid var(--line); border-radius:10px }}
.kpi {{ padding:14px }} .kpi b {{ display:block; font-size:24px }}
.summary {{ padding:16px 18px }}
.item {{ margin-bottom:8px }}
.item summary {{ display:grid; grid-template-columns:28px 56px 1fr 140px 44px; gap:10px; align-items:center;
  padding:12px 14px; cursor:pointer; list-style:none }}
.rank {{ color:var(--muted); font-variant-numeric:tabular-nums }}
.tier {{ font-size:12px; font-weight:600; text-align:center; border-radius:99px; padding:2px 0; color:#fff }}
.t-now {{ background:var(--now) }} .t-next {{ background:var(--next) }} .t-later {{ background:var(--later) }}
.name {{ font-weight:600 }}
.bar {{ height:8px; background:var(--line); border-radius:99px; overflow:hidden }}
.bar span {{ display:block; height:100%; background:var(--accent) }}
.score {{ text-align:right; font-variant-numeric:tabular-nums; font-weight:600 }}
.body {{ padding:0 14px 14px 52px }} .why {{ margin:6px 0 }}
.quotes {{ margin:6px 0 0; padding-left:18px; color:var(--muted); font-size:14px }}
@media (max-width:600px) {{ .item summary {{ grid-template-columns:24px 52px 1fr 40px }} .bar {{ display:none }}
  .body {{ padding-left:14px }} }}
</style></head><body><main>
<h1>Product roadmap from customer feedback</h1>
<div class="muted">Generated {s['generated']} · {channels}</div>
<div class="kpis">
  <div class="kpi"><b>{s['total_feedback']}</b><span class="muted">feedback items</span></div>
  <div class="kpi"><b>{len(s['by_channel'])}</b><span class="muted">channels</span></div>
  <div class="kpi"><b>{s['clusters']}</b><span class="muted">clusters</span></div>
  <div class="kpi"><b>{sum(1 for x in r['roadmap'] if x['tier'] == 'Now')}</b><span class="muted">build-now themes</span></div>
</div>
<h2>Executive summary</h2>
<div class="summary">{escape(r['executive_summary'])}</div>
<h2>Prioritized themes <span class="muted">(click to see evidence)</span></h2>
{''.join(rows)}
<h2>Not on the roadmap</h2>
<ul>{excluded}</ul>
<p class="muted">Score = 30% request volume + 25% distinct accounts + 25% ARR + 10% channel breadth + 10% urgency.
Now ≥ 60 · Next ≥ 35 · Later &lt; 35.</p>
</main></body></html>
"""

**CLI** entry point.

In [ ]:
%%writefile main.py
"""CLI: python main.py data/feedback.csv [--offline] [--out output]"""

import argparse
import os

from roadmap_agent import report
from roadmap_agent.store import FeedbackStore


def main():
    p = argparse.ArgumentParser(description="Cluster customer feedback and build a prioritized roadmap.")
    p.add_argument("input", help="CSV or JSON file with a 'text' column (optional: id, date, channel, customer, plan, arr)")
    p.add_argument("--out", default="output", help="output directory")
    p.add_argument("--offline", action="store_true", help="use the keyword baseline instead of Claude")
    args = p.parse_args()

    store = FeedbackStore.from_file(args.input)
    print(f"Loaded {len(store.feedback)} feedback items from {args.input}")

    if args.offline:
        from roadmap_agent.offline import cluster_offline
        result = cluster_offline(store)
    else:
        if not (os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("ANTHROPIC_AUTH_TOKEN")):
            print("Note: no ANTHROPIC_API_KEY set - the SDK will try your `ant auth login` profile.")
        from roadmap_agent.agent import run_agent
        print("Running Claude agent…")
        result = run_agent(store)

    rep = report.build(store, result)
    for path in report.write_all(rep, args.out):
        print(f"Wrote {path}")
    print("\nTop themes:")
    for x in rep["roadmap"][:5]:
        print(f"  {x['rank']}. [{x['tier']:5}] {x['name']}  (score {x['score']}, "
              f"{x['requests']} requests, {x['accounts']} accounts)")


if __name__ == "__main__":
    main()

**Streamlit front-end.**

In [ ]:
%%writefile app.py
"""Streamlit front-end for the Customer Feedback Clustering & Roadmap Agent.

Run:  streamlit run app.py
"""

import json
import os
import tempfile
from pathlib import Path

import altair as alt
import pandas as pd
import streamlit as st

from roadmap_agent import report
from roadmap_agent.offline import cluster_offline
from roadmap_agent.store import WEIGHTS, FeedbackStore

SAMPLE = Path(__file__).parent / "data" / "feedback.csv"
TIER_COLORS = {"Now": "#dc2626", "Next": "#d97706", "Later": "#64748b"}

st.set_page_config(page_title="Feedback → Roadmap Agent", page_icon="🧭", layout="wide")

# ---------------- sidebar: inputs ----------------
with st.sidebar:
    st.header("1 · Feedback source")
    upload = st.file_uploader("Upload CSV or JSON", type=["csv", "json"],
                              help="Needs a `text` column. Optional: id, date, channel, customer, plan, arr.")
    st.caption("No file? The bundled sample (80 items, 6 channels) is used.")

    st.header("2 · Engine")
    mode = st.radio("Clustering engine", ["Claude agent", "Offline keyword baseline"],
                    help="The baseline needs no API key; it's useful for demos and comparison.")
    api_key = ""
    if mode == "Claude agent":
        # A server-side key (Colab secret / Streamlit Cloud secret) is used on the server and never
        # sent to the browser; visitors can still paste their own key instead.
        server_key = os.environ.get("ANTHROPIC_API_KEY", "")
        user_key = st.text_input("Anthropic API key", type="password",
                                 placeholder="Using the server's key" if server_key else "sk-ant-…",
                                 help="Optional when the app owner has configured a key.")
        api_key = user_key or server_key

    run = st.button("Build roadmap", type="primary", use_container_width=True)

    with st.expander("How scores work"):
        st.markdown("\n".join(f"- **{k}**: {int(v * 100)}%" for k, v in WEIGHTS.items()))
        st.caption("Tiers: Now ≥ 60 · Next ≥ 35 · Later < 35. Scores are computed in code, not by the LLM.")


def load_store() -> FeedbackStore:
    if upload is None:
        return FeedbackStore.from_file(SAMPLE)
    suffix = Path(upload.name).suffix
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(upload.getvalue())
    return FeedbackStore.from_file(tmp.name)


# ---------------- header + raw data preview ----------------
st.title("🧭 Customer Feedback → Product Roadmap")
st.write("An AI agent that categorizes, clusters and counts feature requests from every channel, "
         "then ranks the most-demanded updates.")

try:
    store = load_store()
except Exception as e:  # malformed upload
    st.error(f"Could not read the file: {e}")
    st.stop()

with st.expander(f"Raw feedback ({len(store.feedback)} items)"):
    st.dataframe(pd.DataFrame([vars(f) for f in store.feedback.values()]), use_container_width=True, hide_index=True)

# ---------------- run ----------------
if run:
    if mode == "Claude agent" and not api_key:
        st.sidebar.error("Enter an API key, or switch to the offline baseline.")
        st.stop()
    try:
        if mode == "Claude agent":
            from roadmap_agent.agent import run_agent
            with st.status("Agent is reading and clustering feedback…", expanded=True) as status:
                result = run_agent(store, on_event=lambda line: st.write(line), api_key=api_key)
                status.update(label="Roadmap ready", state="complete", expanded=False)
        else:
            result = cluster_offline(store)
    except Exception as e:
        st.error(f"Run failed: {e}")
        st.stop()
    st.session_state["report"] = report.build(store, result)

rep = st.session_state.get("report")
if not rep:
    st.info("Choose a source and engine in the sidebar, then click **Build roadmap**.")
    st.stop()

# ---------------- results ----------------
s, ranked = rep["summary"], rep["roadmap"]
c1, c2, c3, c4 = st.columns(4)
c1.metric("Feedback items", s["total_feedback"])
c2.metric("Channels", len(s["by_channel"]))
c3.metric("Clusters", s["clusters"])
c4.metric("Build-now themes", sum(1 for x in ranked if x["tier"] == "Now"))

st.subheader("Executive summary")
st.info(rep["executive_summary"] or "—")

tab_rank, tab_detail, tab_channels, tab_export = st.tabs(["Ranking", "Theme evidence", "Channels", "Export"])

with tab_rank:
    df = pd.DataFrame([{
        "Rank": x["rank"], "Tier": x["tier"], "Theme": x["name"], "Score": x["score"],
        "Requests": x["requests"], "Accounts": x["accounts"], "ARR asking": x["arr"],
        "Channels": len(x["channels"]), "Urgent": x["urgent"], "Category": x["category"],
    } for x in ranked])
    chart = alt.Chart(df).mark_bar().encode(
        x=alt.X("Score:Q", scale=alt.Scale(domain=[0, 100])),
        y=alt.Y("Theme:N", sort="-x", title=None, axis=alt.Axis(labelLimit=320)),
        color=alt.Color("Tier:N", scale=alt.Scale(domain=list(TIER_COLORS), range=list(TIER_COLORS.values()))),
        tooltip=list(df.columns),
    ).properties(height=max(240, 34 * len(df)))
    st.altair_chart(chart, use_container_width=True)
    st.dataframe(df, use_container_width=True, hide_index=True,
                 column_config={"ARR asking": st.column_config.NumberColumn(format="$%d"),
                                "Score": st.column_config.ProgressColumn(min_value=0, max_value=100, format="%.1f")})

with tab_detail:
    for x in ranked:
        with st.expander(f"{x['rank']}. [{x['tier']}] {x['name']} — score {x['score']}"):
            st.caption(f"{x['category']} · {x['requests']} requests · {x['accounts']} accounts · "
                       f"${x['arr']:,.0f} ARR · {x['urgent']} urgent · {', '.join(x['channels'])}")
            st.write(x["description"])
            if x.get("rationale"):
                st.markdown(f"**Why:** {x['rationale']}")
            for q in x["quotes"]:
                st.markdown(f"> {q}")
    if rep["excluded"]:
        st.markdown("**Not on the roadmap (bugs & non-requests)**")
        for x in rep["excluded"]:
            st.markdown(f"- {x['name']} ({x['category']}, {x['requests']} items)")

with tab_channels:
    ch = pd.DataFrame(sorted(s["by_channel"].items(), key=lambda kv: -kv[1]), columns=["Channel", "Items"])
    st.altair_chart(alt.Chart(ch).mark_bar().encode(x="Items:Q", y=alt.Y("Channel:N", sort="-x", title=None)),
                    use_container_width=True)

with tab_export:
    st.download_button("Roadmap (Markdown)", report.to_markdown(rep), "roadmap.md", "text/markdown")
    st.download_button("Roadmap (HTML dashboard)", report.to_html(rep), "roadmap.html", "text/html")
    st.download_button("Roadmap (JSON)", json.dumps(rep, indent=2), "roadmap.json", "application/json")

In [ ]:
%%writefile requirements.txt
anthropic>=1.13.0
streamlit>=1.40
pandas
altair

**Sample data** – 80 requests from app store, support tickets, sales calls, NPS, Twitter and forums. Replace with your own or upload a file in the app.

In [ ]:
%%writefile data/feedback.csv
id,date,channel,customer,plan,arr,text
F001,2026-07-02,app_store,anon_1123,free,0,Please add a dark mode! My eyes hurt using this at night.
F002,2026-07-03,support_ticket,Acme Corp,enterprise,120000,"We need SAML SSO with Okta before our security team will approve rollout to 2,000 seats."
F003,2026-07-03,sales_call,Northwind Traders,enterprise,85000,Procurement is blocked on single sign-on. They use Azure AD. Deal at risk without it.
F004,2026-07-04,community_forum,jdoe_pm,pro,1200,Would love a Slack integration so task updates post to our team channel.
F005,2026-07-05,nps_survey,Brightside Labs,pro,4800,Exporting reports to Excel is painful - copy/paste every week. Give us a CSV export.
F006,2026-07-05,app_store,anon_2201,free,0,The app is unusable on the subway. Need offline mode on mobile!
F007,2026-07-06,support_ticket,Globex,enterprise,210000,Our execs want custom dashboards with the KPIs they care about instead of the fixed overview page.
F008,2026-07-07,twitter,@sarah_builds,pro,600,Dark theme when?? Every other tool I use has one.
F009,2026-07-08,support_ticket,Initech,business,24000,Can you add webhooks so we can trigger our internal workflows when a ticket closes?
F010,2026-07-08,nps_survey,Hooli,enterprise,150000,"Love the product, but no SSO is a dealbreaker for renewal. Okta please."
F011,2026-07-09,community_forum,maria_ops,business,9000,Need recurring tasks - we create the same weekly checklist by hand every Monday.
F012,2026-07-10,support_ticket,Umbrella Inc,business,18000,Dashboard takes 12 seconds to load with our 40k records. Very slow.
F013,2026-07-10,app_store,anon_3302,free,0,Would pay for Pro if there was a dark mode.
F014,2026-07-11,sales_call,Stark Industries,enterprise,300000,Security questionnaire requires enforced 2FA for all users and SSO. Both are must-haves.
F015,2026-07-12,community_forum,devon_k,pro,1200,Microsoft Teams integration please. We are not a Slack shop.
F016,2026-07-12,nps_survey,Vandelay Imports,business,12000,Please let us export everything to PDF for our board packs.
F017,2026-07-13,support_ticket,Wayne Enterprises,enterprise,260000,Need a public REST API to sync data with our data warehouse.
F018,2026-07-14,app_store,anon_4410,free,0,App crashes every time I open the calendar view on Android 15.
F019,2026-07-14,twitter,@techlead_amy,pro,600,"Offline support for the iOS app would be huge, I travel a lot."
F020,2026-07-15,support_ticket,Soylent Co,business,15000,Can we build our own reports? The built-in ones don't match how we measure throughput.
F021,2026-07-15,community_forum,raj_m,pro,1200,Templates for recurring projects would save us hours each sprint.
F022,2026-07-16,nps_survey,Massive Dynamic,enterprise,180000,Add two-factor authentication. Our auditors flagged the lack of MFA.
F023,2026-07-17,sales_call,Cyberdyne,enterprise,95000,Prospect requires SCIM provisioning and SAML SSO to move forward.
F024,2026-07-17,app_store,anon_5120,free,0,Dark mode is the only thing missing. 4 stars until then.
F025,2026-07-18,support_ticket,Acme Corp,enterprise,120000,Following up on SSO request - our CISO is asking for a timeline.
F026,2026-07-19,community_forum,lin_design,pro,1200,Bulk export of all project data as CSV would make migrations and backups so much easier.
F027,2026-07-19,twitter,@nightowl_dev,free,0,Begging for a dark UI. Please.
F028,2026-07-20,support_ticket,Pied Piper,business,20000,Slack notifications when someone mentions me would be amazing.
F029,2026-07-21,nps_survey,Brightside Labs,pro,4800,Interface feels sluggish when switching between projects.
F030,2026-07-21,sales_call,Oceanic Airlines,enterprise,140000,They need the product in Spanish and Portuguese for LATAM teams.
F031,2026-07-22,app_store,anon_6001,free,0,Great app! Really helped our small team get organized.
F032,2026-07-23,support_ticket,Globex,enterprise,210000,Need to schedule dashboards to email to leadership every Friday.
F033,2026-07-23,community_forum,ken_w,business,9000,Please support webhooks + an API key per workspace so we can automate.
F034,2026-07-24,support_ticket,Initech,business,24000,Our API integration keeps getting rate limited at 60 req/min. Please raise limits.
F035,2026-07-25,twitter,@pm_jules,pro,600,Need a way to export my data. Feels locked in right now.
F036,2026-07-25,app_store,anon_7123,free,0,No offline mode = I can't use it on flights. Deleting.
F037,2026-07-26,nps_survey,Hooli,enterprise,150000,Two-factor login should be enforceable by admins.
F038,2026-07-27,support_ticket,Umbrella Inc,business,18000,Search is very slow - takes several seconds to return results.
F039,2026-07-27,community_forum,ana_lead,business,9000,Recurring tasks are a basic feature - surprised it's missing.
F040,2026-07-28,sales_call,Tyrell Corp,enterprise,110000,"Lost the evaluation to a competitor mainly because of no Slack/Teams integration."
F041,2026-07-29,support_ticket,Vandelay Imports,business,12000,I was charged twice this month. Please refund.
F042,2026-07-29,app_store,anon_8210,free,0,Night mode please - the white background is blinding.
F043,2026-07-30,nps_survey,Soylent Co,business,15000,Custom charts and the ability to pick which metrics show on the home dashboard.
F044,2026-07-31,community_forum,tom_r,pro,1200,Is there a way to use the app offline on Android? Would really help field teams.
F045,2026-08-01,support_ticket,Wayne Enterprises,enterprise,260000,Still waiting on SSO via Okta. Blocking expansion to 3 more business units.
F046,2026-08-02,twitter,@ux_kate,pro,600,Please localize to German - half our team would prefer it.
F047,2026-08-03,sales_call,Monsters Inc,business,30000,Wants to export Gantt charts to PDF for client presentations.
F048,2026-08-04,app_store,anon_9001,free,0,"Crashes when uploading attachments larger than 10MB on iPhone."
F049,2026-08-04,support_ticket,Pied Piper,business,20000,We'd love a native Microsoft Teams app with notifications.
F050,2026-08-05,nps_survey,Massive Dynamic,enterprise,180000,Admins need to enforce SSO-only login for compliance.
F051,2026-08-06,community_forum,priya_s,pro,1200,Dark mode + high-contrast theme for accessibility please.
F052,2026-08-07,support_ticket,Stark Industries,enterprise,300000,Need SCIM user provisioning tied to our IdP.
F053,2026-08-08,twitter,@remote_ray,pro,600,Recurring reminders would make this my default task app.
F054,2026-08-08,app_store,anon_1010,free,0,"Loading takes forever on 4G, can it be faster?"
F055,2026-08-09,nps_survey,Initech,business,24000,The API is missing endpoints for comments and attachments.
F056,2026-08-10,sales_call,Gringotts,enterprise,220000,Bank requires mandatory MFA and audit logs before purchase.
F057,2026-08-11,support_ticket,Globex,enterprise,210000,Custom report builder with filters and grouping is our #1 ask.
F058,2026-08-12,community_forum,omar_h,business,9000,An Excel export of time tracking data please.
F059,2026-08-13,app_store,anon_1111,free,0,Offline access to my notes would be a game changer.
F060,2026-08-13,twitter,@founder_lee,pro,600,Your onboarding is the best I've seen. Keep it up!
F061,2026-08-14,support_ticket,Oceanic Airlines,enterprise,140000,Can the UI be translated to French for our Montreal office?
F062,2026-08-15,nps_survey,Monsters Inc,business,30000,Integration with Slack would cut our status meetings in half.
F063,2026-08-16,community_forum,zoe_t,pro,1200,Please add a dark mode option on desktop too.
F064,2026-08-17,support_ticket,Hooli,enterprise,150000,Audit log of admin actions is required by our compliance team.
F065,2026-08-18,sales_call,Acme Corp,enterprise,120000,Expansion of 1500 seats contingent on SSO shipping this quarter.
F066,2026-08-19,app_store,anon_1212,free,0,Repeat tasks feature please - daily habits would be perfect here.
F067,2026-08-20,support_ticket,Cyberdyne,enterprise,95000,Need audit logs exportable to our SIEM.
F068,2026-08-21,twitter,@dataguy,pro,600,"API docs are thin and there's no webhook support, so building on you is hard."
F069,2026-08-22,nps_survey,Umbrella Inc,business,18000,Performance with large projects is getting worse every month.
F070,2026-08-23,community_forum,bea_c,pro,1200,How do I change my billing email?
F071,2026-08-24,support_ticket,Soylent Co,business,15000,Please let us save custom report views and share them with the team.
F072,2026-08-25,app_store,anon_1313,free,0,Pleeease dark mode
F073,2026-08-26,sales_call,Wonka Industries,business,40000,Requires Google Workspace SSO for their 300 users.
F074,2026-08-27,support_ticket,Vandelay Imports,business,12000,Export to CSV is missing custom fields.
F075,2026-08-28,community_forum,eli_m,business,9000,Teams integration so approvals can happen in chat.
F076,2026-08-29,nps_survey,Gringotts,enterprise,220000,Two-factor authentication via authenticator app is required.
F077,2026-08-30,app_store,anon_1414,free,0,Offline mode for when I'm on job sites without signal.
F078,2026-08-31,support_ticket,Tyrell Corp,enterprise,110000,Spanish localization needed for our Madrid team.
F079,2026-09-01,twitter,@mobile_max,pro,600,App freezes when scrolling long lists on Android.
F080,2026-09-02,support_ticket,Wayne Enterprises,enterprise,260000,Higher API rate limits and bulk endpoints for our nightly sync.

## 3 · Load the API key from Colab secrets

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded – the Claude agent is enabled.")
except Exception as e:
    print(f"⚠️ No ANTHROPIC_API_KEY secret found ({type(e).__name__}). Use the offline baseline, or paste a key in the app sidebar.")

## 4 · Try it from the notebook
Offline baseline first (instant, free), then the Claude agent.

In [ ]:
!python main.py data/feedback.csv --offline --out output/offline

In [ ]:
# Runs the full Claude agent (takes a few minutes; watch its tool calls stream by).
if os.environ.get("ANTHROPIC_API_KEY"):
    !python main.py data/feedback.csv --out output/agent
else:
    print("Skipped – no API key.")

In [ ]:
from IPython.display import HTML, display
from pathlib import Path
path = Path("output/agent/roadmap.html")
if not path.exists():
    path = Path("output/offline/roadmap.html")
display(HTML(path.read_text()))

## 5 · Launch the Streamlit front-end
Starts Streamlit in the background and opens a free public **Cloudflare tunnel** to it.
Open the `https://….trycloudflare.com` link that is printed. The link stays alive while this cell's runtime is running.

In [ ]:
import re, subprocess, threading, time

!test -f cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared)

subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)
streamlit = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
# Keep draining the tunnel's log so it never blocks.
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
print(f"🚀 Open the app: {url}" if url else "Tunnel failed – see the output above.")

In [ ]:
# Troubleshooting: show the Streamlit server log
!tail -n 30 streamlit.log